# PA3 Task 1 on Kaggle — Full FT runs

Full FT of Qwen3-0.6B needs ~12 GB (FP32 weights + gradients + AdamW states), more than an 8 GB laptop GPU.

**Before running:** right-hand panel → *Session options* → **Accelerator: GPU T4 x2** (one GPU is used) and **Internet: On**.
Then **Save Version → Save & Run All (Commit)**; it runs in the background (12 h limit) and you can close the tab.

Run this notebook twice: once with `METHOD = "full_ft"`, once with `METHOD = "full_ft_l2sp"`.
Recovery checkpoints (~7 GB each) go to `/tmp` because `/kaggle/working` is capped at 20 GB; their sizes are still logged.

In [ ]:
METHOD = "full_ft"   # change to "full_ft_l2sp" for the second run
RUN_EVAL = False     # True: also run lm-eval here (~1 h per condition on a T4); normally done locally

In [ ]:
!git clone -q https://github.com/ahmadhassann96/ATML_PA3.git /kaggle/working/ATML_PA3
%cd /kaggle/working/ATML_PA3
!git log --oneline -1
!pip install -q "transformers>=4.51"
if RUN_EVAL:
    !pip install -q "lm_eval[hf]>=0.4.8"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.environ["PA3_OUT"] = "/kaggle/working/outputs"
os.environ["PA3_CKPT"] = "/tmp/pa3_ckpt"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!python -m task1.data

In [ ]:
# ~1,250 updates; the log prints s/update, peak GPU memory and an ETA every 10 updates
!python -m task1.train --method {METHOD}

In [ ]:
if RUN_EVAL:
    conds = ["full_ft@625", "full_ft"] if METHOD == "full_ft" else ["full_ft_l2sp"]
    for c in conds:
        !python -m task1.evaluate --condition {c} --gen_bs 32 --ll_bs 32

In [ ]:
# one zip to download: exports (FP16 model, FP32 diagnostic weights), logs, resources, config
!rm -f /kaggle/working/outputs/task1/data/train.npz
!cd /kaggle/working/outputs/task1/runs && zip -r -q /kaggle/working/{METHOD}.zip {METHOD}
!ls -lh /kaggle/working/*.zip
!du -sh /kaggle/working/outputs/task1/runs/{METHOD}

**After it finishes:** open the saved version → **Output** tab → download `<METHOD>.zip`, and unzip it so the folder lands at
`outputs/task1/runs/<METHOD>/` in your local PA3 repo. Evaluation, diagnostics and qualitative generation then run locally.